# 原地操作 (In-place Operations)
为了节省显存，PyTorch 允许直接修改张量本身，而不是创建新张量。

规则：所有以 _ 结尾的函数都是原地操作。

- tensor.add_(other): 原地加法（等价于 tensor += other）。
- tensor.mul_(other): 原地乘法（等价于 tensor *= other）。
- tensor.fill_(value): 原地填充。
- tensor.zero_(): 原地清零。

注意：原地操作会破坏计算图，可能影响反向传播（梯度计算），在训练循环中需谨慎使用，但在推理或数据处理阶段非常高效。

## 🛠️ 核心概念：什么是原地操作？
- 非原地操作 (Out-of-place)：例如 y = x + 1。<font color='red'>PyTorch 会分配一块新内存来存储 y，而 x 保持不变。</font>
- 原地操作 (In-place)：例如 x.add_(1)。PyTorch <font color='red'>直接在 x 原本占用的内存地址上修改数值</font>，x 的值被永久改变了。


| 函数 | 功能描述 | 等价代码 | 示例 |
| :--- | :--- | :--- | :--- |
| `tensor.add_(other)` | 原地加法。将 `other` 的值加到 `tensor` 上。 | `tensor += other` | `x.add_(2)` 使 `x` 的每个元素加 2 |
| `tensor.mul_(other)` | 原地乘法。将 `tensor` 乘以 `other`。 | `tensor *= other` | `x.mul_(3)` 使 `x` 的每个元素乘 3 |
| `tensor.fill_(value)` | 原地填充。将 `tensor` 所有元素设为 `value`。 | `tensor[:] = value` | `x.fill_(0)` 将 `x` 全部变为 0 |
| `tensor.zero_()` | 原地清零。将 `tensor` 所有元素设为 0。 | `tensor[:] = 0` | `x.zero_()` 是 `x.fill_(0)` 的特例 |


## ⚖️ 为什么要使用原地操作？（优缺点）
使用带 _ 的函数是一把双刃剑，你需要根据场景权衡：
### ✅ 优点：节省内存

这是原地操作最大的价值。在深度学习训练中，尤其是处理巨大的模型或批量数据时，频繁创建新张量会消耗大量显存（GPU Memory）。原地操作复用了内存，可以显著降低显存占用，防止“显存溢出（OOM）”。

### ❌ 缺点：破坏计算图（梯度计算风险）
PyTorch 使用自动微分（Autograd）来计算梯度。为了计算梯度，系统需要保留前向传播时的中间变量值。
- 风险：如果你在计算图中对某个中间变量使用了原地操作（如 x.add_()），原本的数据就被覆盖了。当反向传播需要用到旧数据时，PyTorch 会报错：
RuntimeError: one of the variables needed for gradient computation has been modified by an inplace operation
- 调试困难：因为数据被直接修改，你无法回溯查看修改前的状态，这增加了调试难度。

## 💡 更多常见的原地操作
除了你提到的，还有很多常用的数学和激活函数也有原地版本：
- 激活函数：tensor.relu_(), tensor.sigmoid_(), tensor.tanh_()
- 数学运算：tensor.exp_(), tensor.sqrt_(), tensor.cos_()
- 线性代数：tensor.t_() (原地转置), tensor.mm_() (原地矩阵乘法)

## 🚀 最佳实践建议
- <font color='red'>默认不使用：在编写模型的前向传播（Forward Pass）代码时，尽量避免使用原地操作，除非你非常确定这不会影响梯度计算。使用 x + y 比 x.add_(y) 更安全。</font>
- <font color='red'>推理和预处理时使用</font>：在模型推理（Inference）阶段（不需要计算梯度）或数据预处理阶段，可以放心使用原地操作来节省内存。
- <font color='red'>使用 torch.no_grad()</font>：如果你必须在训练过程中使用原地操作（例如更新权重），请确保在 with torch.no_grad(): 块内进行，这会告诉 PyTorch 不要追踪这些操作的梯度。

In [ ]:
import torch

x = torch.tensor([1.0, 2.0], requires_grad=True)

# ❌ 危险：在需要梯度的计算图中使用原地操作
# y = x * 2
# y.add_(1)  # 可能导致反向传播报错

# ✅ 安全：在 no_grad 模式下使用
with torch.no_grad():
    x.add_(1)  # 此时不需要计算梯度，原地修改是安全的